Merged the refined clusters from the topdown approach into a unified big one

In [ ]:
%pip install sentence-transformers


In [6]:
import json
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict
import numpy as np
import os 

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load file
with open("/Users/jule/Desktop/CapstoneDATEV25/ontology_output/gcq_ontology_topdown_refined_lj.json", "r", encoding="utf-8") as f:
    clusters = json.load(f)

# If the file is a list instead of dict
if isinstance(clusters, list):
    clusters = {f"Cluster {i}": item for i, item in enumerate(clusters)}

all_classes = []
all_properties = []
class_to_cluster = defaultdict(list)
property_to_cluster = defaultdict(list)
relationships = []

for cluster_id, cluster_data in clusters.items():
    ontology = cluster_data.get("refined_ontology", {})
    
    for c in ontology.get("classes", []):
        if isinstance(c, str):
            all_classes.append(c)
            class_to_cluster[c].append(cluster_id)
    
    for p in ontology.get("properties", []):
        if isinstance(p, str):
            all_properties.append(p)
            property_to_cluster[p].append(cluster_id)
    
    for rel in ontology.get("relationships", []):
        if isinstance(rel, list) and len(rel) == 3:
            relationships.append({
                "source": rel[0],
                "property": rel[1],
                "target": rel[2],
                "cluster": cluster_id
            })


# Deduplicate Classes using embeddings
def deduplicate_terms(terms, threshold=0.85):
    unique = []
    mapping = {}
    embeddings = model.encode(terms, convert_to_tensor=True)
    
    for i, term in enumerate(terms):
        found = False
        for j, u in enumerate(unique):
            sim = util.pytorch_cos_sim(embeddings[i], embeddings[j])[0].item()
            if sim >= threshold:
                mapping[term] = u
                found = True
                break
        if not found:
            unique.append(term)
            mapping[term] = term
    return list(set(mapping.values())), mapping

merged_classes, class_map = deduplicate_terms(all_classes)
merged_properties, property_map = deduplicate_terms(all_properties)

# Rebuild merged relationships
merged_relationships = []
for rel in relationships:
    merged_relationships.append({
        "source": class_map.get(rel["source"], rel["source"]),
        "property": property_map.get(rel["property"], rel["property"]),
        "target": class_map.get(rel["target"], rel["target"]),
        "origin_cluster": rel["cluster"]
    })


# Final merged ontology
merged_ontology = {
    "classes": sorted(set(merged_classes)),
    "properties": sorted(set(merged_properties)),
    "relationships": merged_relationships,
    "metadata": {
        "class_origin": class_to_cluster,
        "property_origin": property_to_cluster
    }
}

# Save
with open("merged_semantic_ontology.json", "w") as f:
    json.dump(merged_ontology, f, indent=2, ensure_ascii=False)

In [7]:
from pyvis.network import Network
import networkx as nx
import json

# Load your merged ontology
with open("merged_semantic_ontology.json", "r", encoding="utf-8") as f:
    ontology = json.load(f)

# Build the graph
G = nx.DiGraph()

for cls in ontology["classes"]:
    G.add_node(cls, label=cls)

for rel in ontology["relationships"]:
    src = rel["source"]
    tgt = rel["target"]
    prop = rel["property"]
    cluster = rel["origin_cluster"]

    G.add_edge(src, tgt, label=prop, title=f"{prop} (from {cluster})")

# Create interactive Pyvis network
net = Network(height="750px", width="100%", notebook=True, directed=True)
net.from_nx(G)
net.show_buttons(filter_=["physics"])  # Optional: physics options

# Show inside notebook
net.show("ontology_graph_notebook.html")


ontology_graph_notebook.html
